In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ugdatalab.plotters.bayesian import plot_posterior, plot_trace


@dataclass(frozen=True)
class MHResult:
    samples: np.ndarray
    log_probs: np.ndarray
    labels: list
    acceptance_rate: float

In [ ]:
x_obs = 1.0
sigma_obs = 0.1


def log_likelihood_1d(mu):
    return -0.5 * ((mu - x_obs) / sigma_obs) ** 2 - np.log(sigma_obs * np.sqrt(2 * np.pi))


def metropolis_hastings_1d(log_prob, theta0, proposal_std, n_steps, seed=42):
    rng = np.random.default_rng(seed)

    samples = np.empty((n_steps, 1))
    log_probs = np.empty(n_steps)

    theta = float(theta0)
    lp_curr = log_prob(theta)
    n_accepted = 0

    for i in range(n_steps):
        theta_prop = theta + rng.normal(0.0, proposal_std)
        lp_prop = log_prob(theta_prop)
        log_alpha = lp_prop - lp_curr

        if np.log(rng.uniform()) < log_alpha:
            theta = theta_prop
            lp_curr = lp_prop
            n_accepted += 1

        samples[i, 0] = theta
        log_probs[i] = lp_curr

    return MHResult(
        samples=samples,
        log_probs=log_probs,
        labels=[r"$\mu$"],
        acceptance_rate=n_accepted / n_steps,
    )

In [ ]:
proposal_grid = [0.05, 0.10, 0.15, 0.18, 0.20, 0.25, 0.30]
proposal_scan = []

for proposal_std in proposal_grid:
    test_chain = metropolis_hastings_1d(
        log_likelihood_1d,
        theta0=0.0,
        proposal_std=proposal_std,
        n_steps=4_000,
        seed=42,
    )
    proposal_scan.append(
        {
            "proposal_std": proposal_std,
            "acceptance_rate": round(test_chain.acceptance_rate, 3),
        }
    )

pd.DataFrame(proposal_scan)

In [ ]:
proposal_std = 0.20
n_steps = 10_000

mh = metropolis_hastings_1d(
    log_likelihood_1d,
    theta0=0.0,
    proposal_std=proposal_std,
    n_steps=n_steps,
    seed=42,
)

mu_samples = mh.samples[:, 0]

print(f"Acceptance rate: {mh.acceptance_rate:.3f}")
print(f"Sample mean of μ: {mu_samples.mean():.4f}   (analytic mean: {x_obs:.4f})")
print(f"Sample std of μ:  {mu_samples.std():.4f}   (analytic std:  {sigma_obs:.4f})")

In [ ]:
def analytic_pdf(mu):
    return np.exp(-0.5 * ((mu - x_obs) / sigma_obs) ** 2) / (sigma_obs * np.sqrt(2 * np.pi))


ax = plot_posterior(mh, param_idx=0, pdf_fn=analytic_pdf)
plt.show()

In [ ]:
axes = plot_trace(mh)
plt.show()